In [ ]:
# Less Salt Less Sugar Monthly Report (launch from 31Aug2026. last for 1 year at least)
#     1. Daily Badge impression by device 
#     2. Daily Brand Page (LMS) Pageview/ Impressions     
#     3. Daily Theme Listing Impressions           --> search --> 少鹽少糖

In [3]:
import datetime 
from dateutil.relativedelta import relativedelta
import pandas as pd
import numpy as np
import pymssql
from shutil import copyfile
from openpyxl import load_workbook
import os
from google.cloud import bigquery
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = 'BQ2.json'

DB_info = {'server':'192.168.61.119:7622', 'user':'BAReporting', 'password':'KeHeCReme8he'}

client = bigquery.Client()

In [4]:
current_date = datetime.date.today()
first_day_of_current_month = datetime.date(current_date.year, current_date.month, 1)
last_day_of_previous_month = first_day_of_current_month - datetime.timedelta(days=1)

month = last_day_of_previous_month.month
year = last_day_of_previous_month.year

str_month = str(month)
if len(str_month)==1:
    str_month = '0'+str_month
str_month

'08'

In [25]:
template_file = 'report_for_BD_template.xlsx'
excel_name = template_file.replace('template.xlsx', '%s.xlsx' % datetime.date.today())
copyfile(template_file, excel_name)

'report_for_BD_2026-09-23.xlsx'

In [ ]:
# 1. Daily Badge impression by device     DONE
#  --> monthly_icon_impression_report_result.ipynb cell 8 少鹽少糖食店 Icon Impression v2
# 
# #少鹽少糖食店 Icon Impression v2

sql = f'''

SELECT date(time) as querydate,platform,count(1) as count_ 
FROM `openrice-production.ORGA.PV_{year}{str_month}*` 
WHERE EventAction = 'impression.poi'
and cast(REGEXP_EXTRACT(lower(EventLabelRaw), r'poiid:(\d+)') as INT64) in  (Select poiid FROM `openrice-production.openrice3.promotionpoi` WHERE PromotionId =11)
group by platform,querydate
    '''

df_big_query = client.query(sql).result().to_dataframe()


<>:12: SyntaxWarning: invalid escape sequence '\d'
<>:12: SyntaxWarning: invalid escape sequence '\d'
C:\Users\lenalee\AppData\Local\Temp\ipykernel_50548\1886240159.py:12: SyntaxWarning: invalid escape sequence '\d'
  '''
C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [42]:
df_big_query

,querydate,platform,count_
0,2026-08-18,ios,365220
1,2026-08-12,android,72891
2,2026-08-20,hms,1171
3,2026-08-30,smart tv,12
4,2026-08-12,tablet,337
...,...,...,...
258,2026-08-01,desktop,20257
259,2026-08-28,tablet,407
260,2026-08-25,android,68198
261,2026-08-31,unknown,3


In [28]:
pivoted_df = df_big_query.pivot(index='querydate', columns='platform', values='count_')
pivoted_df = pivoted_df.fillna(0)
pivoted_df["Web"]=pivoted_df["desktop"]
pivoted_df["Mobile Web"]=pivoted_df["mobile"]
try:
    pivoted_df["Android"]=pivoted_df["android"]+pivoted_df["hms"]
except:
    pivoted_df["Android"]=pivoted_df["android"]
    
pivoted_df["IOS"]=pivoted_df["ios"]
pivoted_df =pivoted_df[["Web","Mobile Web","Android","IOS"]]

pivoted_df.index.name = None
pivoted_df.reset_index(inplace=True)

pivoted_df

platform,index,Web,Mobile Web,Android,IOS
0,2026-08-01,20257,39493,87756,440460
1,2026-08-02,17623,36689,81841,417045
2,2026-08-03,44780,32543,64903,326647
3,2026-08-04,50576,34758,73025,363109
4,2026-08-05,50139,34689,74189,368825
5,2026-08-06,51556,34699,76806,376998
6,2026-08-07,54856,37493,79905,403192
7,2026-08-08,30397,42042,90812,453735
8,2026-08-09,32759,37463,92661,460509
9,2026-08-10,55797,34911,71397,357474


In [ ]:
# with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
# #     book = load_workbook(excel_name)
# #     writer.book = book
# #     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
#     pivoted_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=7, header=None, index=False)

In [30]:
# 2. Daily Brand Page (LMS) Pageview       no date, group by web & app
# 17:37:14|| view.SR1.Promotion| CityID:0;PromotionID:13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C12%2C13%2C13%2C13%2C13%2C13%2C13;;Lang:zh_TW;Ver:7.20.4; sn:hk.LMS2.35336.tab.-990.1


sql = f"""
with theme_list as (
    select platform
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (LOWER(EventAction) = 'view.sr1.promotion'
    AND ((LOWER(EventLabelRaw) LIKE '%lms%') or (LOWER(EventLabelRaw) LIKE '%promotionid:13%')))
)

select platform, count(1) as count
from theme_list
group by platform
"""

df_big_query_2 = client.query(sql).result().to_dataframe()

web = df_big_query_2.query("platform=='mobile' | platform=='desktop'  ")["count"].sum()
app = df_big_query_2.query("platform=='android' | platform=='ios' | platform=='hms' ")["count"].sum()
temp_df = pd.DataFrame({'date':[f'{year}-{str_month}'],'web':[web],'app':[app]})

temp_df

# with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
# #     book = load_workbook(excel_name)
# #     writer.book = book
# #     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
#     temp_df.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=42, header=None, index=False)


C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,date,web,app
0,2026-08,444989,3494589


In [31]:
df_big_query_2

,platform,count
0,bot,12047
1,android,695663
2,ios,2789620
3,unknown,73
4,mobile,269672
5,desktop,175317
6,tablet,1483
7,smart tv,11
8,hms,9306


In [15]:
# 2. Daily Brand Page (LMS) Pageview   hv date, group by devices
# 17:37:14|| view.SR1.Promotion| CityID:0;PromotionID:13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C13%2C12%2C13%2C13%2C13%2C13%2C13%2C13;;Lang:zh_TW;Ver:7.20.4; sn:hk.LMS2.35336.tab.-990.1
sql = f"""
    select date(time) as querydate, platform, count(1) as count
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (
        LOWER(EventAction) = 'view.sr1.promotion'
        AND (
            (LOWER(EventLabelRaw) LIKE '%lms%') or (LOWER(EventLabelRaw) LIKE '%promotionid:13%')         
            )
        )
    group by platform, querydate
    """

# condition for app and web

df_big_query_2_2 = client.query(sql).result().to_dataframe()


C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [16]:
df_big_query_2_2

,querydate,platform,count
0,2026-08-20,hms,110
1,2026-08-10,desktop,16
2,2026-08-07,android,41666
3,2026-08-26,hms,114
4,2026-08-16,mobile,10
...,...,...,...
190,2026-08-04,android,36585
191,2026-08-11,ios,27862
192,2026-08-04,hms,533
193,2026-08-16,ios,32890


In [ ]:
pivoted_df_2 = df_big_query_2_2.pivot(index='querydate', columns='platform', values='count')
pivoted_df_2 = pivoted_df_2.fillna(0)
expected_platforms = ["desktop", "mobile", "ios", "android", "hms"]
pivoted_df_2 = pivoted_df_2.reindex(columns=expected_platforms, fill_value=0)

pivoted_df_2["Web"]=pivoted_df_2["desktop"] + pivoted_df_2["mobile"]
pivoted_df_2["App"]=pivoted_df_2["ios"] + pivoted_df_2["android"] + pivoted_df_2["hms"]
pivoted_df_2 =pivoted_df_2[["Web","App"]]

pivoted_df_2.index.name = None
pivoted_df_2.reset_index(inplace=True)

pivoted_df_2

platform,index,Web,App
0,2026-08-01,22264,230679
1,2026-08-02,23076,221412
2,2026-08-03,28612,166870
3,2026-08-04,39766,182390
4,2026-08-05,43145,185048
5,2026-08-06,44078,191122
6,2026-08-07,38687,209464
7,2026-08-08,432,106437
8,2026-08-09,285,87188
9,2026-08-10,185,35312


In [ ]:
# with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
# #     book = load_workbook(excel_name)
# #     writer.book = book
# #     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
#     pivoted_df_2.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=42, header=None, index=False)

In [ ]:
# 3. Daily Theme Listing Impressions       adv search
# 17:38:23|| or.search.adv| CityID:0;geo:22.2915161%2C114.2081815;LndID:35336;Page:1;sr:lmsSr1;Lang:zh_TW;Ver:7.20.4; sn:hk.AdvSearch

sql = f"""
    select date(time) as querydate, platform, count(1) as count
    from `openrice-production.ORGA.PV_{year}{str_month}*`
    WHERE (
        LOWER(EventAction) = 'or.search.adv'
        AND (
            (LOWER(EventLabelRaw) LIKE '%35336%')
            or
            (LOWER(EventData) LIKE '%35336%')
            or
            (LOWER(EventLabelRaw) LIKE '%lms%')
            )
        )
    group by platform, querydate
    """

# condition for web amd app

df_big_query_3 = client.query(sql).result().to_dataframe()


C:\Users\lenalee\AppData\Roaming\Python\Python313\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [10]:
#df_big_query_3

In [19]:
pivoted_df_3 = df_big_query_3.pivot(index='querydate', columns='platform', values='count')
pivoted_df_3 = pivoted_df_3.fillna(0)

expected_platforms = ["desktop", "mobile", "ios", "android", "hms"]
pivoted_df_3 = pivoted_df_3.reindex(columns=expected_platforms, fill_value=0)
pivoted_df_3["Web"]= pivoted_df_3["desktop"] + pivoted_df_3["mobile"]
pivoted_df_3["App"]=pivoted_df_3["ios"] + pivoted_df_3["android"] + pivoted_df_3["hms"]
pivoted_df_3 =pivoted_df_3[["Web","App"]]


pivoted_df_3.index.name = None
pivoted_df_3.reset_index(inplace=True)

pivoted_df_3


platform,index,Web,App
0,2026-08-01,0,1345
1,2026-08-02,0,1263
2,2026-08-03,0,846
3,2026-08-04,0,880
4,2026-08-05,0,1113
5,2026-08-06,0,911
6,2026-08-07,0,1064
7,2026-08-08,0,1393
8,2026-08-09,0,1311
9,2026-08-10,1,991


In [ ]:
# with pd.ExcelWriter(excel_name, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
# #     book = load_workbook(excel_name)
# #     writer.book = book
# #     writer.sheets = dict((ws.title, ws) for ws in book.worksheets)
#     pivoted_df_3.to_excel(writer, sheet_name='reduction of salt.sugar', startrow=77, header=None, index=False)